# Data Merging and Preprocessing
This notebook merges the raw datasets, strictly caps all Security (SE) samples to 200 rows globally to achieve perfect balance for both binary and sub_NFR labels, removes duplicates and noise, and splits the data.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split

data_dir = '../data/raw'
processed_dir = '../data/processed'

# 1. PROMISE-relabeled-NICE
df_nice = pd.read_csv(os.path.join(data_dir, 'PROMISE-relabeled-NICE.csv'))
nfr_cols = ['Availability (A)', 'Fault Tolerance (FT)', 'Legal (L)', 'Look & Feel (LF)', 
            'Maintainability (MN)', 'Operability (O)', 'Performance (PE)', 'Portability (PO)', 
            'Scalability (SC)', 'Security (SE)', 'Usability (US)', 'Other (OT)']

def get_nice_sub_nfr(row):
    for col in nfr_cols:
        if row[col] == 1:
            if '(' in col and ')' in col:
                return col.split('(')[1].split(')')[0]
            return col
    return None

df_nice['text'] = df_nice['RequirementText']
df_nice['label_binary'] = df_nice.apply(lambda r: 'FR' if r['IsFunctional'] == 1 and r['IsQuality'] == 0 else 'NFR', axis=1)
df_nice['sub_NFR'] = df_nice.apply(get_nice_sub_nfr, axis=1)
df_nice['source'] = 'NICE'
nice_final = df_nice[['text', 'label_binary', 'sub_NFR', 'source']].copy()

# 2. promise_exp
df_exp = pd.read_csv(os.path.join(data_dir, 'promise_exp.csv'))
df_exp['text'] = df_exp['RequirementText']
df_exp['label_binary'] = df_exp['class'].apply(lambda x: 'FR' if str(x).strip().upper() == 'F' else 'NFR')
df_exp['sub_NFR'] = df_exp['class']
df_exp['source'] = 'promise_exp'
exp_final = df_exp[['text', 'label_binary', 'sub_NFR', 'source']].copy()

# 3. dcai24
df_dcai = pd.read_excel(os.path.join(data_dir, 'dcai24_src_dataset.xlsx'))
df_dcai['text'] = df_dcai['Requirement']
df_dcai['sub_NFR'] = df_dcai['Specific_Type']

def get_dcai_binary(row):
    t = str(row['Type']).strip().upper()
    s = str(row['Specific_Type']).strip().upper()
    if t == 'FR' or s == 'NONSE':
        return 'FR'
    return 'NFR'

df_dcai['label_binary'] = df_dcai.apply(get_dcai_binary, axis=1)
df_dcai['sub_NFR'] = df_dcai['sub_NFR'].replace({'1': 'L', 'l': 'L', 'NONSEPO': 'PO', 'NONSE': 'F'})
df_dcai['source'] = 'dcai24'
dcai_final = df_dcai[['text', 'label_binary', 'sub_NFR', 'source']].copy()

# Combine all WITHOUT EARS
merged = pd.concat([nice_final, exp_final, dcai_final], ignore_index=True)

# Deduplication
merged = merged.drop_duplicates(subset=['text'], keep='first')

# Check noise
nan_text = merged['text'].isna() | (merged['text'].astype(str).str.strip() == '')
nfr_missing_sub = (merged['label_binary'] == 'NFR') & (merged['sub_NFR'].isna() | (merged['sub_NFR'].astype(str).str.strip() == ''))

# Delete all noise
merged_clean = merged[~(nan_text | nfr_missing_sub)].copy()

# Fix FR missing sub_NFR
merged_clean.loc[(merged_clean['label_binary'] == 'FR') & (merged_clean['sub_NFR'].isna()), 'sub_NFR'] = 'F'

# CAPPING ALL SE GLOBALLY TO 200 ROWS
se_df = merged_clean[merged_clean['sub_NFR'] == 'SE']
non_se_df = merged_clean[merged_clean['sub_NFR'] != 'SE']
if len(se_df) > 200:
    se_df = se_df.sample(n=200, random_state=42)

merged_clean = pd.concat([non_se_df, se_df], ignore_index=True)
merged_clean = merged_clean.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total rows after capping ALL SE to 200: {len(merged_clean)}")

# Train/Val/Test Split
X = merged_clean[['text', 'sub_NFR', 'source']]
y = merged_clean['label_binary']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

val_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio, stratify=y_train_val, random_state=42
)

train_df = X_train.copy()
train_df['label_binary'] = y_train
val_df = X_val.copy()
val_df['label_binary'] = y_val
test_df = X_test.copy()
test_df['label_binary'] = y_test

train_df.to_csv(os.path.join(processed_dir, 'train.csv'), index=False)
val_df.to_csv(os.path.join(processed_dir, 'val.csv'), index=False)
test_df.to_csv(os.path.join(processed_dir, 'test.csv'), index=False)
print("Saved updated train/val/test splits.")


Total rows after capping ALL SE to 200: 2272
Saved updated train/val/test splits.
